<a href="https://colab.research.google.com/github/yoshiagent/vibration-colab/blob/main/notebooks/SDOF_Response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 單自由度自由與步階響應 SDOF Free and Step Response

對照網頁：[振動學｜單自由度自由與步階響應](https://yoshiagent.github.io/vibration/SD.html)

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `sys2ndfree.m` | $\ddot y+2\zeta\omega_n\dot y+\omega_n^2y=0$ 三種阻尼（欠、臨界、過阻尼）的自由響應公式 | 第 1 節 |
| `order2step.m` | 標準二階系統 $\omega_n^2/(s^2+2\zeta\omega_n s+\omega_n^2)$ 的步階響應 | 第 2 節 |

每個公式都和 `scipy.integrate.solve_ivp` 的數值積分比對。只用 Colab 內建的 numpy、scipy、plotly。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go


def show(fig, title, xlab="t (s)", ylab="y(t)", h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def ord2(wn, z):
    """MATLAB Control Toolbox 的 ord2：num = 1, den = [1 2*z*wn wn^2]"""
    return [1.0], [1.0, 2 * z * wn, wn**2]


def free_numeric(z, wn, y0, v0, t):
    sol = solve_ivp(lambda t, x: [x[1], -2 * z * wn * x[1] - wn**2 * x[0]], (t[0], t[-1]), [y0, v0], t_eval=t, rtol=1e-11, atol=1e-12)
    return sol.y[0]

## 1. 自由響應的三種情形（`sys2ndfree.m`）

```matlab
z=0.3; wn=2; wd=wn*sqrt(1-z^2); y0=1; v0=0; t=0:0.001:10;
rt=roots([1 2*z*wn wn^2])
if (z<1) & (z>-1)                       % 欠阻尼
  c1=y0;  c2=(v0+z*wn*y0)/wd;
  y=exp(-z*wn*t).*(c2*sin(wd*t)+c1*cos(wd*t));       % 另畫包絡線 ±sqrt(c1^2+c2^2)*exp(-z*wn*t)
elseif (z==1) | (z==-1)                 % 臨界阻尼
  c1=y0;  c2=v0+wn*z*y0;  y=(c1+c2*t).*exp(-wn*z*t);
elseif (z>1) | (z<-1)                   % 過阻尼
  z1=sqrt(z^2-1);
  c1=(v0+(z+z1)*wn*y0)/(2*wn*z1);  c2=(-v0-(z-z1)*wn*y0)/(2*wn*z1);
  y=exp(-z*wn*t).*(c1*exp(z1*wn*t)+c2*exp(-z1*wn*t));
end
```
（原檔第一行多了一個 `<html>`，是存檔時混入的，MATLAB 執行會出錯；其餘照錄。）

這裡的欠阻尼 $c_1=y_0$、$c_2=(v_0+\zeta\omega_n y_0)/\omega_d$ 是正確的；網頁講義圖上把兩者寫反，已加編註更正。

In [2]:
def sys2ndfree(z, wn, y0, v0, t):
    """sys2ndfree.m 的三種情形（回傳 y 與包絡線，沒有包絡線時為 None）"""
    if -1 < z < 1:
        wd = wn * np.sqrt(1 - z**2)
        c1, c2 = y0, (v0 + z * wn * y0) / wd
        env = np.sqrt(c1**2 + c2**2) * np.exp(-z * wn * t)
        return np.exp(-z * wn * t) * (c2 * np.sin(wd * t) + c1 * np.cos(wd * t)), env
    if abs(z) == 1:
        c1, c2 = y0, v0 + wn * z * y0
        return (c1 + c2 * t) * np.exp(-wn * z * t), None
    z1 = np.sqrt(z**2 - 1)
    c1 = (v0 + (z + z1) * wn * y0) / (2 * wn * z1)
    c2 = (-v0 - (z - z1) * wn * y0) / (2 * wn * z1)
    return np.exp(-z * wn * t) * (c1 * np.exp(z1 * wn * t) + c2 * np.exp(-z1 * wn * t)), None


z, wn, y0, v0 = 0.3, 2.0, 1.0, 0.0
t = np.arange(0, 10 + 1e-9, 0.001)
print("rt = roots([1 2*z*wn wn^2]) =", np.round(np.roots([1, 2 * z * wn, wn**2]), 4))
y, env = sys2ndfree(z, wn, y0, v0, t)
s = slice(None, None, 10)                    # 計算用 dt = 0.001（同原程式），畫圖每 10 點取一點
fig = go.Figure([go.Scatter(x=t[s], y=y[s], name="y(t)"),
                 go.Scatter(x=t[s], y=env[s], name="envelope", line=dict(color="red", dash="dot")),
                 go.Scatter(x=t[s], y=-env[s], showlegend=False, line=dict(color="red", dash="dot"))])
show(fig, "Free response, zeta = 0.3, wn = 2, y0 = 1, v0 = 0 (sys2ndfree.m)")

rt = roots([1 2*z*wn wn^2]) = [-0.6+1.9079j -0.6-1.9079j]


改變阻尼比比較（同一組 $\omega_n=2$、$y_0=1$、$v_0=0$）。每條曲線都與數值積分比對：

In [3]:
fig = go.Figure()
tt = np.linspace(0, 10, 2001)
for z in (0.0, 0.1, 0.3, 0.7, 1.0, 2.0, 5.0):
    y, _ = sys2ndfree(z, wn, 1.0, 0.0, tt)
    err = np.max(np.abs(y - free_numeric(z, wn, 1.0, 0.0, tt)))
    print(f"zeta = {z}: max |formula - solve_ivp| = {err:.1e}")
    fig.add_trace(go.Scatter(x=tt, y=y, name=f"zeta = {z}"))
show(fig, "Free response for several damping ratios")
# 非零初速度也驗證一次（三種情形的 c1、c2 都用到 v0）
for z in (0.3, 1.0, 2.0):
    y, _ = sys2ndfree(z, wn, 0.5, -3.0, tt)
    print(f"y0 = 0.5, v0 = -3, zeta = {z}: max |formula - solve_ivp| = {np.max(np.abs(y - free_numeric(z, wn, 0.5, -3.0, tt))):.1e}")

zeta = 0.0: max |formula - solve_ivp| = 2.6e-11
zeta = 0.1: max |formula - solve_ivp| = 6.3e-12
zeta = 0.3: max |formula - solve_ivp| = 3.2e-12
zeta = 0.7: max |formula - solve_ivp| = 2.6e-12
zeta = 1.0: max |formula - solve_ivp| = 7.1e-13
zeta = 2.0: max |formula - solve_ivp| = 1.5e-12
zeta = 5.0: max |formula - solve_ivp| = 2.7e-12


y0 = 0.5, v0 = -3, zeta = 0.3: max |formula - solve_ivp| = 4.4e-12
y0 = 0.5, v0 = -3, zeta = 1.0: max |formula - solve_ivp| = 1.8e-12
y0 = 0.5, v0 = -3, zeta = 2.0: max |formula - solve_ivp| = 5.4e-13


## 2. 步階響應與時域規格（`order2step.m`）

```matlab
wn=5; z=0.3;
[num den]=ord2(wn,z);       % num = 1, den = [1 2*z*wn wn^2]
g=tf(num*wn^2,den);         % G(s) = wn^2/(s^2+2 z wn s+wn^2)
step(g)
```
網頁講義推導出：
$y(t)=1-\dfrac{e^{-\zeta\omega_n t}}{\sqrt{1-\zeta^2}}\sin(\omega_d t+\phi)$，$\phi=\cos^{-1}\zeta$，
$t_r=\dfrac{\pi-\phi}{\omega_d}$、$t_p=\dfrac{\pi}{\omega_d}$、$M_p=e^{-\zeta\pi/\sqrt{1-\zeta^2}}$、$t_s\approx 4/(\zeta\omega_n)$（2%）。

In [4]:
wn, z = 5.0, 0.3
num, den = ord2(wn, z)
g = signal.TransferFunction(np.array(num) * wn**2, den)
t = np.linspace(0, 3, 30001)
_, y = signal.step(g, T=t)
wd, phi = wn * np.sqrt(1 - z**2), np.arccos(z)
yf = 1 - np.exp(-z * wn * t) / np.sqrt(1 - z**2) * np.sin(wd * t + phi)
print(f"step: max |scipy step - closed form| = {np.max(np.abs(y - yf)):.1e}")
tr = t[np.argmax(y >= 1)]
tp, Mp = t[np.argmax(y)], y.max() - 1
inside = np.abs(y - 1) <= 0.02
ts = t[np.where(~inside)[0][-1] + 1]
print(f"t_r = {tr:.4f} s  (formula (pi - acos z)/wd = {(np.pi - phi) / wd:.4f})")
print(f"t_p = {tp:.4f} s  (formula pi/wd = {np.pi / wd:.4f})")
print(f"M_p = {Mp * 100:.2f} %  (formula exp(-z pi/sqrt(1-z^2)) = {np.exp(-z * np.pi / np.sqrt(1 - z**2)) * 100:.2f} %)")
print(f"t_s(2%) = {ts:.4f} s  (approximation 4/(z wn) = {4 / (z * wn):.4f})")
L = np.log(Mp) / np.pi
print(f"zeta from M_p: sqrt((ln Mp/pi)^2/(1+(ln Mp/pi)^2)) = {np.sqrt(L**2 / (1 + L**2)):.4f}")
s = slice(None, None, 20)
fig = go.Figure([go.Scatter(x=t[s], y=y[s], name="step response"),
                 go.Scatter(x=[tp], y=[1 + Mp], mode="markers+text", text=[f"Mp = {Mp * 100:.1f}%"], textposition="top right", name="peak"),
                 go.Scatter(x=[0, 3], y=[1.02, 1.02], line=dict(color="gray", dash="dot"), name="±2%"),
                 go.Scatter(x=[0, 3], y=[0.98, 0.98], line=dict(color="gray", dash="dot"), showlegend=False)])
show(fig, "Step response, wn = 5, zeta = 0.3 (order2step.m)", ylab="Amplitude")

step: max |scipy step - closed form| = 7.9e-14
t_r = 0.3933 s  (formula (pi - acos z)/wd = 0.3932)
t_p = 0.6587 s  (formula pi/wd = 0.6587)
M_p = 37.23 %  (formula exp(-z pi/sqrt(1-z^2)) = 37.23 %)
t_s(2%) = 2.2461 s  (approximation 4/(z wn) = 2.6667)
zeta from M_p: sqrt((ln Mp/pi)^2/(1+(ln Mp/pi)^2)) = 0.3000


$t_s\approx 4/(\zeta\omega_n)$ 只是包絡線 $e^{-\zeta\omega_n t}/\sqrt{1-\zeta^2}$ 進入 2% 的近似（還忽略了 $1/\sqrt{1-\zeta^2}$）；
實際安定時間要看響應最後一次離開 ±2% 帶的時刻，兩者通常相差一成以內。